# FraudTrace AI - 06: Multi-Class Contradiction Detection Engine

> **DISCLAIMER: SYNTHETIC DATA ONLY**
> All data used in this notebook is synthetically generated for research, testing, and demonstration purposes. No real police records, personal identification, or banking transactions are included. This system does not make legal guilt determinations; it reconstructs and correlates evidentiary relationships.

### Purpose
Detect forensic contradictions (amount mismatch, timestamp drift, missing data, duplicate records) across evidence items.

| Attribute | Description |
| :--- | :--- |
| **Input** | Pairs or clusters of correlated evidence items |
| **Output** | Standardized forensic contradiction classification JSON |
| **Target Environment** | Google Colab (CPU / GPU compatible) & Local |
| **Package** | `ml.src` |


In [ ]:
# ====================================================================
# GOOGLE COLAB ENVIRONMENT SETUP
# ====================================================================
import os
import sys

# Clone repository if running inside Google Colab
if 'google.colab' in str(get_ipython()):
    if not os.path.exists('FraudTrace.AI'):
        !git clone https://github.com/Cyberdude441/FraudTrace.AI.git
    %cd FraudTrace.AI
    !pip install -r ml/requirements-colab.txt

# Ensure repository root is on sys.path for local and Colab execution
repo_root = os.path.abspath('.')
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

print(f"[OK] Environment initialized. Working directory: {os.getcwd()}")


## 1. Contradiction Taxonomy
The contradiction engine classifies evidence relationships into 6 strict forensic categories:
1. `CORROBORATED`: Multiple independent sources agree across amount, time, and entities.
2. `CONFLICTING`: Sources exhibit direct irreconcilable factual discrepancies (e.g. ₹50,000 vs ₹48,000).
3. `PARTIALLY_CORROBORATED`: Agreement on key fields with minor non-blocking ambiguities.
4. `MISSING_DATA`: Critical forensic fields (e.g. timestamp or amount) are absent.
5. `TIMESTAMP_INCONSISTENCY`: Amounts and entities agree, but timestamps diverge beyond plausible drift.
6. `POSSIBLE_DUPLICATE`: Cryptographic hash or text match indicating duplicated records rather than independent corroboration.

In [ ]:
from ml.src.correlation.contradiction import ContradictionEngine
import json

engine = ContradictionEngine()

scenarios = [
    ("Scenario 1: Amount Mismatch (50k vs 48k)", [
        {"id": "E1", "type": "FIR", "source": "Complainant", "amount": 50000.0, "timestamp": "2024-03-15T10:00:00Z"},
        {"id": "E2", "type": "BANK_STATEMENT", "source": "Bank Core", "amount": 48000.0, "timestamp": "2024-03-15T10:02:00Z"}
    ]),
    ("Scenario 2: Timestamp Drift (2 Hours)", [
        {"id": "E1", "type": "SMS", "source": "Gateway", "amount": 50000.0, "timestamp": "2024-03-15T10:00:00Z"},
        {"id": "E2", "type": "BANK_STATEMENT", "source": "Bank Core", "amount": 50000.0, "timestamp": "2024-03-15T12:00:00Z"}
    ]),
    ("Scenario 3: Missing Required Timestamp", [
        {"id": "E1", "type": "RECEIPT", "source": "Store Pos", "amount": 50000.0, "timestamp": None},
        {"id": "E2", "type": "BANK_STATEMENT", "source": "Bank Core", "amount": 50000.0, "timestamp": "2024-03-15T10:00:00Z"}
    ]),
    ("Scenario 4: Duplicate Record", [
        {"id": "E1", "type": "SCREENSHOT", "source": "Phone User", "amount": 50000.0, "timestamp": "2024-03-15T10:00:00Z", "sha256": "abc123hash"},
        {"id": "E2", "type": "SCREENSHOT", "source": "Phone Backup", "amount": 50000.0, "timestamp": "2024-03-15T10:00:00Z", "sha256": "abc123hash"}
    ]),
    ("Scenario 5: Corroborated Evidence (3 Independent Sources)", [
        {"id": "E1", "type": "FIR", "source": "Complainant", "amount": 50000.0, "timestamp": "2024-03-15T10:00:00Z"},
        {"id": "E2", "type": "BANK_STATEMENT", "source": "Bank Core", "amount": 50000.0, "timestamp": "2024-03-15T10:01:00Z"},
        {"id": "E3", "type": "SMS_GATEWAY", "source": "Telecom Gateway", "amount": 50000.0, "timestamp": "2024-03-15T10:01:30Z"}
    ])
]

for title, ev_list in scenarios:
    res = engine.evaluate_event(ev_list)
    print(f"\n{title}")
    print(f" - Status        : {res['status']}")
    print(f" - Severity      : {res['severity']}")
    print(f" - Discrepancies : {len(res['discrepancies'])}")
    print(f" - Explanation   : {res['explanation']}")
